In [1]:
import matplotlib.pyplot as plt
import numpy as np
import string
import os
import time
from pickle import dump, load
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, get_file
from tensorflow.keras.layers import Add 
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout
from tqdm.notebook import tqdm as tqdm
from PIL import Image
# tqdm().pandas()


### Load any type of document. Whether it is an image or a text file

In [2]:
def load_doc(filename):
    file = open(filename, 'r')
    text = file.read()
    file.close()
    return text

In [3]:
def all_img_captions(filename):
    file = load_doc(filename)
    captions = file.split('\n')
    descriptions = {}
    for caption in captions[:-1]:
        image, caption = caption.split('\t')
        if image[:-2] not in descriptions:
            descriptions[image[:-2]] = [caption]
        else:
            descriptions[image[:-2]].append(caption)
    return descriptions

In [4]:
def clean_text(captions):
    table = str.maketrans('','',string.punctuation)
    for image, caps in captions.items():
        for i, image_caption in enumerate(caps):
            image_caption.replace("-","")
            desc = image_caption.split()
            desc = [word.lower() for word in desc]
            desc = [word.translate(table) for word in desc]
            desc = [word for word in desc if len(word) > 1]
            desc = [word for word in desc if word.isalpha()]

            image_caption = ' '.join(desc)
            captions[image][i] = image_caption
    return captions

In [5]:
def generate_inMemory_vocabulary(descriptions):
    vocab = set()
    for key in descriptions:
        [vocab.update(d.split()) for d in descriptions[key]]
    return vocab

In [6]:
def save_descriptions(descriptions,filename):
    lines = list()
    for key, desc_list in descriptions.items():
        for desc in desc_list:
            lines.append(key+'\t'+desc)
    data = '\n'.join(lines)
    file = open(filename,'w')
    file.write(data)
    file.close

In [7]:
dataset_text_dir_name = "data/processed/training"
dataset_images_dir_name = "data/raw"

In [8]:
token_filename = dataset_text_dir_name + '/Flickr8k.token.txt'
descriptions = all_img_captions(token_filename)
print("Length of descriptions: ", len(descriptions))

i = 0
for key in descriptions:
    if i==5:
        break
    print(key)
    i = i + 1

Length of descriptions:  8092
1000268201_693b08cb0e.jpg
1001773457_577c3a7d70.jpg
1002674143_1b742ab4b8.jpg
1003163366_44323f5815.jpg
1007129816_e794419615.jpg


In [9]:
cleaned_description = clean_text(descriptions)
cleaned_description

{'1000268201_693b08cb0e.jpg': ['child in pink dress is climbing up set of stairs in an entry way',
  'girl going into wooden building',
  'little girl climbing into wooden playhouse',
  'little girl climbing the stairs to her playhouse',
  'little girl in pink dress going into wooden cabin'],
 '1001773457_577c3a7d70.jpg': ['black dog and spotted dog are fighting',
  'black dog and tricolored dog playing with each other on the road',
  'black dog and white dog with brown spots are staring at each other in the street',
  'two dogs of different breeds looking at each other on the road',
  'two dogs on pavement moving toward each other'],
 '1002674143_1b742ab4b8.jpg': ['little girl covered in paint sits in front of painted rainbow with her hands in bowl',
  'little girl is sitting in front of large painted rainbow',
  'small girl in the grass plays with fingerpaints in front of white canvas with rainbow on it',
  'there is girl with pigtails sitting in front of rainbow painting',
  'young 

In [10]:
vocabulary = generate_inMemory_vocabulary(cleaned_description)
print("Length of Vocab: ", len(vocabulary))

Length of Vocab:  8763


In [11]:
save_descriptions(cleaned_description,"generated/descriptions/descriptions.txt")

In [12]:
def download_with_retry(url, filename, max_retries = 3):
    for attempt in range(max_retries):
        try:
            return get_file(filename, url)
        except Exception as e:
            if attempt == max_retries - 1 :
                raise e
            print("Download attempt failed")
            print("Reason" + str(e))
            time.sleep(3)

In [56]:
xception_weights_url = "https://storage.googleapis.com/tensorflow/keras-applications/xception/xception_weights_tf_dim_ordering_tf_kernels.h5"
weights_save_url = 'pre-trained/xception'
weights_path = download_with_retry(xception_weights_url, "xception_weights.h5")
print(weights_path)

C:\Users\thane\.keras\datasets\xception_weights.h5


In [13]:
def extract_features(directory):
    model = Xception(include_top = False, pooling = 'avg', weights = 'imagenet')
    features = {}
    valid_images_extension = ['.jpg','.jpeg','.png']
    for img in os.listdir(directory):
        extension = os.path.splitext(img)[1].lower()
        if extension not in valid_images_extension:
            continue
        filename = directory + '/' + img
        image = Image.open(filename)
        image = image.resize((299,299))
        image = np.expand_dims(image, axis= 0)
        image = image/127.5
        image = image - 1.0

        feature = model.predict(image)
        print(feature)
        features[img] = feature
    print("Extracted features: ",features)
    return features


In [58]:
features = extract_features(dataset_images_dir_name)
dump(features, open("features.p","wb"))

1/1 [==============================] - 1s 1s/step
[[0.4734095  0.01730895 0.07334225 ... 0.08557959 0.02102284 0.23765522]]
1/1 [==============================] - 1s 524ms/step
[[0.00158196 0.1111413  0.00037429 ... 0.26504433 0.35281467 0.05871542]]
1/1 [==============================] - 1s 515ms/step
[[0.         0.02493176 0.01557379 ... 0.         0.         0.10194056]]


KeyboardInterrupt: 

In [14]:
features

NameError: name 'features' is not defined

In [15]:
def load_photos(filename):
    file = load_doc(filename)
    photos = file.split('\n')[:-1]
    photos_present = [photo for photo in photos if os.path.exists(os.path.join(dataset_images_dir_name, photo))]
    return photos_present

In [16]:
def load_clean_descriptions(filename, photos):
    file = load_doc(filename)
    descriptions = {}
    for line in file.split('\n'):
        words = line.split()
        if len(words) < 1:
            continue
        image,image_caption = words[0], words[1:]
        if image in photos:
            descriptions[image] = []
            desc = '<start> '+" ".join(image_caption) + ' <end>'
            descriptions[image].append(desc)
    return descriptions

In [17]:
def load_features(train_images):
    all_features = load(open("features.p", 'rb'))
    print("All Features: ", all_features)
    features = {k: all_features[k] for k in train_images}
    return features

In [18]:
token_filename = dataset_text_dir_name + '/Flickr_8k.trainImages.txt'

# train_images is a list of all the training images that are present in the dataset
train_images = load_photos(token_filename)

# print("train_images: ", train_images[:5])
print("Length of training photos: ", len(train_images))

train_descriptions = load_clean_descriptions("generated/descriptions/descriptions.txt", train_images)
print("Length of training descriptions: ", len(train_descriptions))
train_features = load_features(train_images)
print("Length of training features: ", len(train_features))

Length of training photos:  6000
Length of training descriptions:  6000
All Features:  {'1000268201_693b08cb0e.jpg': array([[0.4734095 , 0.01730895, 0.07334225, ..., 0.08557959, 0.02102284,
        0.23765522]], dtype=float32), '1001773457_577c3a7d70.jpg': array([[0.00158196, 0.1111413 , 0.00037429, ..., 0.26504433, 0.35281467,
        0.05871542]], dtype=float32), '1002674143_1b742ab4b8.jpg': array([[0.        , 0.02493176, 0.01557379, ..., 0.        , 0.        ,
        0.10194056]], dtype=float32), '1003163366_44323f5815.jpg': array([[0.14568025, 0.00272414, 0.27775782, ..., 0.17016979, 0.11957434,
        0.09415711]], dtype=float32), '1007129816_e794419615.jpg': array([[0.        , 0.12444555, 0.7391587 , ..., 0.00390426, 0.00997243,
        0.50171375]], dtype=float32), '1007320043_627395c3d8.jpg': array([[0.04138522, 0.        , 0.01274429, ..., 0.00944679, 0.6420155 ,
        0.04793232]], dtype=float32), '1009434119_febe49276a.jpg': array([[0.        , 0.        , 0.02635757,

In [19]:
def dic_to_list(descriptions):
    all_desc = []
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

In [20]:
def create_tokenizer(descriptions):
    desc_list = dic_to_list(descriptions)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(desc_list)
    return tokenizer

In [21]:
tokenizer  = create_tokenizer(train_descriptions)

In [22]:
dump(tokenizer, open("generated/serialized/tokenizer/tokenizer.p","wb"))

In [23]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size: ", vocab_size)

Vocabulary Size:  3836


In [24]:
def max_length_descriptions(descriptions):
    desc_list = dic_to_list(descriptions)
    return max(len(d.split()) for d in desc_list)

In [25]:
max_length = max_length_descriptions(train_descriptions)
print("Max Length of any description: ", max_length)

Max Length of any description:  34


In [26]:
def create_sequences(tokenizer, max_length, description_list, feature, vocab_size):
    input_img, input_seq, output_word = [], [], []
    for description in description_list:
        seq = tokenizer.texts_to_sequences([description])[0]
        for i in range(1, len(seq)):
            in_seq, out_seq = seq[:i], seq[i]
            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
            out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
            input_img.append(feature)
            input_seq.append(in_seq)
            output_word.append(out_seq)
    return np.array(input_img), np.array(input_seq), np.array(output_word)

In [27]:
def data_generator(descriptions, features, tokenizer, max_length, vocab_size):
    def generator():
        while True:
            for key, description_list in descriptions.items():
                feature = features[key][0]
                input_img, input_seq, output_word = create_sequences(tokenizer, max_length, description_list, feature, vocab_size)
                for i in range(len(input_img)):
                    yield {'input_1': input_img[i], 'input_2': input_seq[i]}, output_word[i] 
    output_signature = (
            {
                'input_1': tf.TensorSpec(shape=(2048,), dtype=tf.float32), 
                'input_2': tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
            },
            tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32)  
        )    
    dataset = tf.data.Dataset.from_generator(generator, output_signature=output_signature)
    return dataset.batch(64)

In [28]:
dataset = data_generator(train_descriptions, train_features, tokenizer, max_length, vocab_size)
dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
for (a,b) in dataset.take(1):
    print(a['input_1'].shape)
    print(a['input_2'].shape)
    print(b.shape)

(64, 2048)
(64, 34)
(64, 3836)


In [29]:
def define_model(vocab_size, max_length):

    #CNN model from which we will extract the features from the image
    # from 2048 nodes to 256 nodes
    inputs1 = Input(shape=(2048,), name = 'input_1')
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    #LSTM model which will take the caption as input and output a vector of 256 nodes
    inputs2 = Input(shape=(max_length,), name= 'input_2' )
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)

    # Decoder model which will take the output from both the CNN and LSTM and output
    # a probability distribution over the vocabulary

    decoder1 = Add()([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model


In [30]:
model = define_model(vocab_size, max_length)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 34)]         0           []                               
                                                                                                  
 input_1 (InputLayer)           [(None, 2048)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 34, 256)      982016      ['input_2[0][0]']                
                                                                                                  
 dropout (Dropout)              (None, 2048)         0           ['input_1[0][0]']                
                                                                                              

In [69]:
model = define_model(vocab_size, max_length)
model.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 34)]         0           []                               
                                                                                                  
 input_1 (InputLayer)           [(None, 2048)]       0           []                               
                                                                                                  
 embedding_4 (Embedding)        (None, 34, 256)      982016      ['input_2[0][0]']                
                                                                                                  
 dropout_8 (Dropout)            (None, 2048)         0           ['input_1[0][0]']                
                                                                                            

In [70]:
epochs = 20
steps_per_epoch = 5
# os.mkdir("models")
for i in range(epochs):
    dataset = data_generator(train_descriptions, train_features, tokenizer, max_length, vocab_size)
    model.fit(dataset, epochs=10, steps_per_epoch = steps_per_epoch, verbose=1)
    model.save("models/checkpoints/epoch_" + str(i) + ".h5")

Epoch 1/10
5/5 [==============================] - 9s 1s/step - loss: 8.1280
Epoch 2/10
5/5 [==============================] - 6s 1s/step - loss: 7.3324
Epoch 3/10
5/5 [==============================] - 6s 1s/step - loss: 6.2116
Epoch 4/10
5/5 [==============================] - 6s 1s/step - loss: 6.7460
Epoch 5/10
5/5 [==============================] - 6s 1s/step - loss: 6.1348
Epoch 6/10
5/5 [==============================] - 6s 1s/step - loss: 6.1935
Epoch 7/10
5/5 [==============================] - 6s 1s/step - loss: 6.2555
Epoch 8/10
5/5 [==============================] - 6s 1s/step - loss: 6.1070
Epoch 9/10
5/5 [==============================] - 6s 1s/step - loss: 5.9218
Epoch 10/10
5/5 [==============================] - 6s 1s/step - loss: 6.0502
Epoch 1/10
5/5 [==============================] - 6s 1s/step - loss: 5.3329
Epoch 2/10
5/5 [==============================] - 6s 1s/step - loss: 5.4680
Epoch 3/10
5/5 [==============================] - 6s 1s/step - loss: 5.0116
Epoch 4/10


In [31]:
def extract_features(filename, model):
    try:
        image = Image.open(filename)

    except : 
        print("Error Occured")

    image = image.resize((299,299))
    image = np.array(image)
    if image.shape[2] == 4 :
        image = image[... , 3]
    image = np.expand_dims(image, axis = 0)
    image = image / 127.5
    image = image - 1.0
    feature = model.predict(image)
    return feature


In [32]:
def word_for_id(integer, tokenizer):
    for word,index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [33]:
def generate_description(model, tokenizer, image, max_length):
    in_text = 'start'
    for i in range(max_length):
        sequence = tokenizer.text_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen = max_length)
        prediction = model.predict([image, sequence], verbose = 0)
        prediction = np.argmax(prediction)
        word = word_for_id(prediction, tokenizer)
        if word is None:
            break
        in_text = in_text +" " + word
        if word == 'end':
            break
    return in_text

In [37]:
import argparse

In [39]:
arg_parser = argparse.ArgumentParser()
arg_parser.add_argument('-i','--image', required = True, help = 'Image')
args = vars(arg_parser.parse_args())
img_path = args['image']

usage: ipykernel_launcher.py [-h] -i IMAGE
ipykernel_launcher.py: error: the following arguments are required: -i/--image


SystemExit: 2

e:\ML Projects\-20M-Image-Caption-Generator-Using-Deep-CNN-and-LSTM\.venv\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [34]:
max_length = 32
tokenizer = load(open('generated/serialized/tokenizer/tokenizer.p',"rb"))
vocab_size = len(tokenizer.word_index) + 1

In [35]:
model = define_model(vocab_size = vocab_size, max_length = max_length)
model.load_weights('models/checkpoints/epoch_19.h5')

In [36]:
xception_model = Xception(include_top = False, pooling = "avg")

In [ ]:
photo = extract_features(img_path, xception_model)

In [ ]:
img = Image.open(img_path)

In [ ]:
description = generate_description(model, tokenizer,photo, max_length)

print(description)